In [1]:
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset
import torchvision.transforms as transforms

class IndianaPaDiMDataset(Dataset):
    def __init__(self, csv_path, resize=256, crop=224):
        self.df = pd.read_csv(csv_path).reset_index(drop=True)

        self.transform = transforms.Compose([
            transforms.Resize((resize, resize)),
            transforms.CenterCrop(crop),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = row["path"]
        label = int(row["label"])
        uid = int(row["uid"])

        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)

        return image, label, uid

In [2]:
from torch.utils.data import DataLoader

train_dataset = IndianaPaDiMDataset("train_padim.csv")
val_dataset = IndianaPaDiMDataset("val_padim.csv")
test_dataset = IndianaPaDiMDataset("test_padim.csv")

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=False, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=0)

images, labels, uids = next(iter(train_loader))

print(images.shape)   
print(labels[:5])
print(uids[:5])

torch.Size([16, 3, 224, 224])
tensor([0, 0, 0, 0, 0])
tensor([3224, 2024, 1182, 2089, 1689])


In [3]:
import pandas as pd
import numpy as np
from PIL import Image
from torch.utils.data import Dataset
import torchvision.transforms as transforms
import torch

class IndianaPaDiMDataset(Dataset):
    def __init__(self, csv_path, resize=256, crop=224):
        self.df = pd.read_csv(csv_path).reset_index(drop=True)
        self.crop = crop

        self.transform = transforms.Compose([
            transforms.Resize((resize, resize)),
            transforms.CenterCrop(crop),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = row["path"]
        label = int(row["label"])

        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)

        # Indiana 沒有 pixel-level mask
        dummy_mask = torch.zeros((1, self.crop, self.crop), dtype=torch.int64)

        return image, label, dummy_mask

In [4]:
train_dataset = IndianaPaDiMDataset("train_padim.csv")
test_dataset = IndianaPaDiMDataset("test_padim.csv")

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=False, num_workers=0)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

In [9]:
import os
import random
import pickle
from collections import OrderedDict

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, f1_score
from scipy.spatial.distance import mahalanobis
from scipy.ndimage import gaussian_filter

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.models import wide_resnet50_2, resnet18


# =========================
# Config
# =========================
TRAIN_CSV = "train_padim.csv"
VAL_CSV = "val_padim.csv"
TEST_CSV = "test_padim.csv"

SAVE_PATH = "./indiana_padim_result"
ARCH = "wide_resnet50_2"   # choose from ["resnet18", "wide_resnet50_2"]
BATCH_SIZE = 32
RESIZE = 256
CROP = 224
SEED = 1024
NUM_WORKERS = 0
SMOOTH_SIGMA = 4


# =========================
# Device setup
# =========================
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")


# =========================
# Dataset
# =========================
class IndianaPaDiMDataset(Dataset):
    def __init__(self, csv_path, resize=256, crop=224):
        self.df = pd.read_csv(csv_path).reset_index(drop=True)
        self.crop = crop

        self.transform = transforms.Compose([
            transforms.Resize((resize, resize)),
            transforms.CenterCrop(crop),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row["path"]
        label = int(row["label"])   # 0 = normal, 1 = abnormal

        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)

        dummy_mask = torch.zeros((1, self.crop, self.crop), dtype=torch.int64)
        return image, label, dummy_mask


# =========================
# Utilities
# =========================
def embedding_concat(x, y):
    B, C1, H1, W1 = x.size()
    _, C2, H2, W2 = y.size()
    s = int(H1 / H2)

    x = F.unfold(x, kernel_size=s, dilation=1, stride=s)
    x = x.view(B, C1, -1, H2, W2)

    z = torch.zeros(B, C1 + C2, x.size(2), H2, W2, device=x.device)
    for i in range(x.size(2)):
        z[:, :, i, :, :] = torch.cat((x[:, :, i, :, :], y), 1)

    z = z.view(B, -1, H2 * W2)
    z = F.fold(z, kernel_size=s, output_size=(H1, W1), stride=s)
    return z


def set_seed(seed=1024):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if use_cuda:
        torch.cuda.manual_seed_all(seed)


def build_model():
    if ARCH == "resnet18":
        model = resnet18(weights="DEFAULT")
        t_d = 448
        d = 100
    elif ARCH == "wide_resnet50_2":
        model = wide_resnet50_2(weights="DEFAULT")
        t_d = 1792
        d = 550
    else:
        raise ValueError(f"Unsupported ARCH: {ARCH}")

    model.to(device)
    model.eval()
    idx = torch.tensor(random.sample(range(0, t_d), d)).to(device)
    return model, idx


def register_hooks(model, outputs):
    def hook(module, input, output):
        outputs.append(output)

    h1 = model.layer1[-1].register_forward_hook(hook)
    h2 = model.layer2[-1].register_forward_hook(hook)
    h3 = model.layer3[-1].register_forward_hook(hook)
    return h1, h2, h3


def remove_hooks(hooks):
    for h in hooks:
        h.remove()


def get_dataloader(csv_path):
    dataset = IndianaPaDiMDataset(csv_path, resize=RESIZE, crop=CROP)
    dataloader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )
    return dataloader


def extract_embeddings(model, dataloader):
    outputs = []
    hooks = register_hooks(model, outputs)

    feat_outputs = OrderedDict([("layer1", []), ("layer2", []), ("layer3", [])])
    gt_list = []

    for (x, y, _) in tqdm(dataloader, desc="Feature extraction"):
        gt_list.extend(y.cpu().numpy())

        with torch.no_grad():
            _ = model(x.to(device))

        for k, v in zip(feat_outputs.keys(), outputs):
            feat_outputs[k].append(v.detach().cpu())

        outputs.clear()

    remove_hooks(hooks)

    for k, v in feat_outputs.items():
        feat_outputs[k] = torch.cat(v, 0)

    return feat_outputs, np.asarray(gt_list)


def make_embedding_vectors(feat_outputs, idx):
    embedding_vectors = feat_outputs["layer1"]
    for layer_name in ["layer2", "layer3"]:
        embedding_vectors = embedding_concat(embedding_vectors, feat_outputs[layer_name])

    embedding_vectors = torch.index_select(embedding_vectors, 1, idx.cpu())
    return embedding_vectors


def fit_train_distribution(model, idx, train_csv, train_feature_filepath):
    if os.path.exists(train_feature_filepath):
        print(f"Loading train distribution from: {train_feature_filepath}")
        with open(train_feature_filepath, "rb") as f:
            train_outputs = pickle.load(f)
        return train_outputs

    print("Extracting train features...")
    train_dataloader = get_dataloader(train_csv)
    train_feat_outputs, _ = extract_embeddings(model, train_dataloader)

    embedding_vectors = make_embedding_vectors(train_feat_outputs, idx)

    B, C, H, W = embedding_vectors.size()
    embedding_vectors = embedding_vectors.view(B, C, H * W)

    mean = torch.mean(embedding_vectors, dim=0).numpy()
    cov = torch.zeros(C, C, H * W).numpy()
    I = np.identity(C)

    print("Fitting Gaussian per spatial location...")
    for i in tqdm(range(H * W), desc="Train Gaussian fitting"):
        cov[:, :, i] = np.cov(embedding_vectors[:, :, i].numpy(), rowvar=False) + 0.01 * I

    train_outputs = [mean, cov]

    with open(train_feature_filepath, "wb") as f:
        pickle.dump(train_outputs, f)

    print(f"Saved train distribution to: {train_feature_filepath}")
    return train_outputs


def score_dataset(model, idx, train_outputs, csv_path, save_csv_name):
    dataloader = get_dataloader(csv_path)
    feat_outputs, gt_list = extract_embeddings(model, dataloader)

    embedding_vectors = make_embedding_vectors(feat_outputs, idx)

    B, C, H, W = embedding_vectors.size()
    embedding_vectors = embedding_vectors.view(B, C, H * W).numpy()

    dist_list = []
    print(f"Computing Mahalanobis distance for {csv_path} ...")
    for i in tqdm(range(H * W), desc=f"Scoring {os.path.basename(csv_path)}"):
        mean = train_outputs[0][:, i]
        cov_inv = np.linalg.inv(train_outputs[1][:, :, i])
        dist = [mahalanobis(sample[:, i], mean, cov_inv) for sample in embedding_vectors]
        dist_list.append(dist)

    dist_list = np.array(dist_list).transpose(1, 0).reshape(B, H, W)

    dist_list = torch.tensor(dist_list, dtype=torch.float32)
    score_map = F.interpolate(
        dist_list.unsqueeze(1),
        size=CROP,
        mode="bilinear",
        align_corners=False
    ).squeeze(1).numpy()

    for i in range(score_map.shape[0]):
        score_map[i] = gaussian_filter(score_map[i], sigma=SMOOTH_SIGMA)

    max_score = score_map.max()
    min_score = score_map.min()
    scores = (score_map - min_score) / (max_score - min_score + 1e-12)

    img_scores = scores.reshape(scores.shape[0], -1).max(axis=1)

    auroc = roc_auc_score(gt_list, img_scores)
    fpr, tpr, _ = roc_curve(gt_list, img_scores)

    result_df = pd.read_csv(csv_path).copy()
    result_df["anomaly_score"] = img_scores

    save_csv_path = os.path.join(SAVE_PATH, save_csv_name)
    result_df.to_csv(save_csv_path, index=False)

    print(f"Saved scores to: {save_csv_path}")
    print(f"AUROC on {os.path.basename(csv_path)}: {auroc:.4f}")

    return {
        "labels": gt_list,
        "scores": img_scores,
        "auroc": auroc,
        "fpr": fpr,
        "tpr": tpr,
        "df": result_df,
    }


def find_best_threshold_from_validation(val_labels, val_scores):
    precision, recall, thresholds = precision_recall_curve(val_labels, val_scores)

    # thresholds 長度比 precision/recall 少 1
    f1_scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-8)

    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx]
    best_val_f1 = f1_scores[best_idx]

    return best_threshold, best_val_f1, f1_scores, precision, recall, thresholds


def evaluate_test_f1(test_labels, test_scores, threshold):
    y_pred = (test_scores > threshold).astype(int)
    test_f1 = f1_score(test_labels, y_pred)
    return test_f1, y_pred


def save_metrics_text(filepath, metrics_dict):
    with open(filepath, "w") as f:
        for k, v in metrics_dict.items():
            f.write(f"{k}: {v}\n")


def main():
    os.makedirs(SAVE_PATH, exist_ok=True)
    set_seed(SEED)

    model, idx = build_model()

    temp_dir = os.path.join(SAVE_PATH, f"temp_{ARCH}")
    os.makedirs(temp_dir, exist_ok=True)
    train_feature_filepath = os.path.join(temp_dir, "train_indiana_frontal.pkl")

    # 1) fit train distribution
    train_outputs = fit_train_distribution(
        model=model,
        idx=idx,
        train_csv=TRAIN_CSV,
        train_feature_filepath=train_feature_filepath
    )

    # 2) score validation
    print("\n===== Validation scoring =====")
    val_result = score_dataset(
        model=model,
        idx=idx,
        train_outputs=train_outputs,
        csv_path=VAL_CSV,
        save_csv_name="val_scores.csv"
    )

    # 3) score test
    print("\n===== Test scoring =====")
    test_result = score_dataset(
        model=model,
        idx=idx,
        train_outputs=train_outputs,
        csv_path=TEST_CSV,
        save_csv_name="test_scores.csv"
    )

    # 4) threshold from validation
    best_threshold, best_val_f1, _, _, _, _ = find_best_threshold_from_validation(
        val_result["labels"],
        val_result["scores"]
    )

    # 5) test F1 using validation-selected threshold
    test_f1, y_pred = evaluate_test_f1(
        test_result["labels"],
        test_result["scores"],
        best_threshold
    )

    test_result["df"]["pred_label"] = y_pred
    test_result["df"].to_csv(os.path.join(SAVE_PATH, "test_scores.csv"), index=False)

    print("\n===== Final Metrics =====")
    print(f"Validation AUROC: {val_result['auroc']:.4f}")
    print(f"Test AUROC: {test_result['auroc']:.4f}")
    print(f"Best validation threshold: {best_threshold:.6f}")
    print(f"Best validation F1: {best_val_f1:.4f}")
    print(f"Test F1: {test_f1:.4f}")

    # 6) save metrics
    metrics = {
        "ARCH": ARCH,
        "BATCH_SIZE": BATCH_SIZE,
        "RESIZE": RESIZE,
        "CROP": CROP,
        "SMOOTH_SIGMA": SMOOTH_SIGMA,
        "Validation_AUROC": f"{val_result['auroc']:.6f}",
        "Test_AUROC": f"{test_result['auroc']:.6f}",
        "Best_Validation_Threshold": f"{best_threshold:.6f}",
        "Best_Validation_F1": f"{best_val_f1:.6f}",
        "Test_F1": f"{test_f1:.6f}",
    }
    metrics_path = os.path.join(SAVE_PATH, "metrics.txt")
    save_metrics_text(metrics_path, metrics)
    print(f"Saved metrics to: {metrics_path}")

    # 7) plot ROC curves
    plt.figure(figsize=(6, 6))
    plt.plot(val_result["fpr"], val_result["tpr"], label=f"Val AUROC = {val_result['auroc']:.4f}")
    plt.plot(test_result["fpr"], test_result["tpr"], label=f"Test AUROC = {test_result['auroc']:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("Indiana Frontal PaDiM ROC Curves")
    plt.legend(loc="lower right"import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, precision_recall_curve, f1_score

val_df = pd.read_csv("indiana_padim_result/val_scores.csv")
test_df = pd.read_csv("indiana_padim_result/test_scores.csv")

val_labels = val_df["label"].values
val_scores = val_df["anomaly_score"].values
test_labels = test_df["label"].values
test_scores = test_df["anomaly_score"].values

val_auroc = roc_auc_score(val_labels, val_scores)
test_auroc = roc_auc_score(test_labels, test_scores)

precision, recall, thresholds = precision_recall_curve(val_labels, val_scores)
f1_scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-8)

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_val_f1 = f1_scores[best_idx]

y_pred = (test_scores > best_threshold).astype(int)
test_f1 = f1_score(test_labels, y_pred)

print(f"Validation AUROC: {val_auroc:.4f}")
print(f"Test AUROC: {test_auroc:.4f}")
print(f"Best validation threshold: {best_threshold:.6f}")
print(f"Best validation F1: {best_val_f1:.4f}")
print(f"Test F1: {test_f1:.4f}"))
    plt.tight_layout()

    roc_path = os.path.join(SAVE_PATH, "roc_curve_val_test.png")
    plt.savefig(roc_path, dpi=150)
    plt.close()
    print(f"Saved ROC curve to: {roc_path}")


if __name__ == "__main__": 
    main()

Loading train distribution from: ./indiana_padim_result/temp_wide_resnet50_2/train_indiana_frontal.pkl

===== Validation scoring =====


Feature extraction:   0%|                                | 0/23 [00:00<?, ?it/s]/opt/miniconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Feature extraction: 100%|███████████████████████| 23/23 [01:34<00:00,  4.10s/it]


Computing Mahalanobis distance for val_padim.csv ...


Scoring val_padim.csv: 100%|████████████████| 3136/3136 [00:37<00:00, 82.83it/s]


Saved scores to: ./indiana_padim_result/val_scores.csv
AUROC on val_padim.csv: 0.6190

===== Test scoring =====


Feature extraction:   0%|                                | 0/51 [00:00<?, ?it/s]/opt/miniconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Feature extraction: 100%|███████████████████████| 51/51 [03:32<00:00,  4.17s/it]


Computing Mahalanobis distance for test_padim.csv ...


Scoring test_padim.csv: 100%|███████████████| 3136/3136 [01:16<00:00, 41.14it/s]


Saved scores to: ./indiana_padim_result/test_scores.csv
AUROC on test_padim.csv: 0.6367

===== Final Metrics =====
Validation AUROC: 0.6190
Test AUROC: 0.6367
Best validation threshold: 0.185443
Best validation F1: 0.9120
Test F1: 0.9207
Saved metrics to: ./indiana_padim_result/metrics.txt
Saved ROC curve to: ./indiana_padim_result/roc_curve_val_test.png


In [10]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, precision_recall_curve, f1_score

val_df = pd.read_csv("indiana_padim_result/val_scores.csv")
test_df = pd.read_csv("indiana_padim_result/test_scores.csv")

val_labels = val_df["label"].values
val_scores = val_df["anomaly_score"].values
test_labels = test_df["label"].values
test_scores = test_df["anomaly_score"].values

val_auroc = roc_auc_score(val_labels, val_scores)
test_auroc = roc_auc_score(test_labels, test_scores)

precision, recall, thresholds = precision_recall_curve(val_labels, val_scores)
f1_scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-8)

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_val_f1 = f1_scores[best_idx]

y_pred = (test_scores > best_threshold).astype(int)
test_f1 = f1_score(test_labels, y_pred)

print(f"Validation AUROC: {val_auroc:.4f}")
print(f"Test AUROC: {test_auroc:.4f}")
print(f"Best validation threshold: {best_threshold:.6f}")
print(f"Best validation F1: {best_val_f1:.4f}")
print(f"Test F1: {test_f1:.4f}")

Validation AUROC: 0.6190
Test AUROC: 0.6367
Best validation threshold: 0.185443
Best validation F1: 0.9120
Test F1: 0.9207


In [ ]:
val_df = pd.read_csv("val_padim.csv")
val_scores = ...

precision, recall, thresholds = precision_recall_curve(val_labels, val_scores)
f1 = 2 * precision * recall / (precision + recall + 1e-8)

best_threshold = thresholds[np.argmax(f1)]